# Full.ipynb — 完整 Car+Motor+Pass 直接建拓樸 → 特徵化 → 訓練

流程：
1. `preprocess(target='全部')` 取完整資料（含時間），先依時間切成 70% train / 30% test。
2. 以 train-only Jenks 建立醫院距離、城鎮距離、鄰近人口三個類別型地理特徵；原始經緯度只用來計算距離，不直接進 Mapper 或模型。
3. 原事故欄位與三個地理特徵一起建 train-only one-hot 與 MCA(n=5) lens；test 只做欄位對齊與 `transform`。
4. 用 `overlap=3, interval=8`（CubicalCover n_intervals=8, overlap_frac=0.3）只在 train 建 Mapper 圖。
5. 從 train Mapper 提取 cycle、core、centrality、articulation、bridge 與距離特徵；test 只繼承最近 train observation 的拓樸摘要。
6. distance/coreness/centrality 類特徵使用 train-only Jenks；三個布林特徵保留 0/1。全程不使用死亡標籤，也不建立 occupancy matrix。


In [ ]:
%load_ext autoreload
%autoreload 2
import os, pickle, time
import numpy as np
import pandas as pd
import networkx as nx
import prince

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
version3_path = os.path.join(parent_dir, "Version3")
os.chdir(version3_path)   # 借用 Version3 的 tdamapper / utils / Data

from sklearn.cluster import AgglomerativeClustering
from tdamapper.core_old import MapperAlgorithm
from tdamapper.cover import CubicalCover
from tdamapper.clustering import FailSafeClustering
from utils.preprocess import preprocess, process_other

# 載入新版 models.py（Version3/utils 沒有 models，用檔案路徑載）
import importlib.util
_mp = os.path.join(parent_dir, "Models", "utils", "models.py")
_spec = importlib.util.spec_from_file_location("models_new", _mp)
models_new = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(models_new)
_ev = os.path.join(parent_dir, "Models", "utils", "evaluate.py")
_spec2 = importlib.util.spec_from_file_location("evaluate", _ev)
evaluate = importlib.util.module_from_spec(_spec2); _spec2.loader.exec_module(evaluate)

dataA2 = pd.read_csv("./Data/A2.csv", low_memory=False)
dataA1 = pd.read_csv("./Data/A1.csv")

In [ ]:
select_lst = [
    # 月份是為了篩選每個月2萬筆
    '發生月份',

    '天候名稱', '光線名稱', 
    '道路類別-第1當事者-名稱', '速限-第1當事者', 
    '路面狀況-路面鋪裝名稱', '路面狀況-路面狀態名稱', '路面狀況-路面缺陷名稱',
    '道路障礙-障礙物名稱', '道路障礙-視距品質名稱', '道路障礙-視距名稱',
    '號誌-號誌種類名稱', '號誌-號誌動作名稱',
    '車道劃分設施-分道設施-快車道或一般車道間名稱', '車道劃分設施-分道設施-快慢車道間名稱', '車道劃分設施-分道設施-路面邊線名稱',
    '當事者屬-性-別名稱', '當事者事故發生時年齡',
    '保護裝備名稱', '行動電話或電腦或其他相類功能裝置名稱',
    # '肇事逃逸類別名稱-是否肇逃',
    '死亡受傷人數',

    # 大類別
    '道路型態大類別名稱', '事故位置大類別名稱',
    '車道劃分設施-分向設施大類別名稱',
    '事故類型及型態大類別名稱', '當事者區分-類別-大類別名稱-車種', '當事者行動狀態大類別名稱',
    '車輛撞擊部位大類別名稱-最初', '車輛撞擊部位大類別名稱-其他',
    
    # 子類別
    # '道路型態子類別名稱', '事故位置子類別名稱', '事故類型及型態子類別名稱', '肇因研判子類別名稱-主要',
    # '當事者區分-類別-子類別名稱-車種', '當事者行動狀態子類別名稱', '車輛撞擊部位子類別名稱-最初',
    # '車輛撞擊部位子類別名稱-其他', '肇因研判子類別名稱-個別',
]

# OSM Jenks 是 preprocess 後才生成的欄位，會在後面加入 mapper_select_lst；原始經緯度不加入。
full_dataA1 = preprocess(dataA1, target='全部', lst=select_lst, add_features=True)
full_dataA2 = preprocess(dataA2, target='全部', lst=select_lst, add_features=True)
# add_features=True 會多出：對造車種/當事人數/有無大型車、時段/深夜/週末/季節（無洩漏）
mapper_numpy, rbind_data, dummy_data, death, injuried, date_info = process_other(
    full_dataA1, full_dataA2, downsample=False, en=False, return_time=True)
print('rbind_data:', rbind_data.shape, '| dummy:', dummy_data.shape)

In [ ]:
# ==== 先依時間切分；後續 Jenks、one-hot、MCA、Mapper 都只 fit train ====
_n = len(rbind_data)
_order_index = date_info.sort_values(['發生日期', '發生時間']).index
_order_pos = rbind_data.index.get_indexer(_order_index)
if (_order_pos < 0).any():
    raise ValueError('date_info 與 rbind_data 的 index 無法對齊')
_n_train = _n - int(np.ceil(_n * 0.3))
train_pos = _order_pos[:_n_train]
test_pos = _order_pos[_n_train:]
assert len(np.unique(_order_pos)) == _n
assert np.intersect1d(train_pos, test_pos).size == 0
assert np.array_equal(np.sort(np.r_[train_pos, test_pos]), np.arange(_n))
print('time split | train:', len(train_pos), '| test:', len(test_pos))


In [ ]:
# ==== OSM 地理特徵：原始經緯度只用來算距離；所有補值與 Jenks 分界只用 train ====
import geopandas as gpd
from sklearn.neighbors import BallTree
import jenkspy

OSM_DIR = "/Users/wangqiqian/Desktop/QGIS/taiwan-251009-free.shp"
R_KM = 6371.0088

def _latlon_rad(gdf):
    g = gdf.copy()
    if g.crs is not None and g.crs.to_epsg() != 4326:   # 若非 WGS84 就轉（本資料已是 4326，不需 transform）
        g = g.to_crs(4326)
    pts = g.geometry.representative_point()             # 點層回傳點本身；面層回傳內部代表點
    return np.radians(np.c_[pts.y.values, pts.x.values])

# 醫院：點層 + 面層
_h1 = gpd.read_file(f"{OSM_DIR}/gis_osm_pois_free_1.shp");   _h1 = _h1[_h1['fclass'] == 'hospital']
_h2 = gpd.read_file(f"{OSM_DIR}/gis_osm_pois_a_free_1.shp"); _h2 = _h2[_h2['fclass'] == 'hospital']
hosp_rad = np.vstack([_latlon_rad(_h1), _latlon_rad(_h2)])

# 城鎮（city/town）與 有人口的地點
_places = gpd.read_file(f"{OSM_DIR}/gis_osm_places_free_1.shp")
city_rad = _latlon_rad(_places[_places['fclass'].isin(['city', 'town'])])
_pop = _places[_places['population'] > 0]
pop_rad = _latlon_rad(_pop)
pop_val = _pop['population'].to_numpy()

# 事故座標（來自 date_info，與 rbind_data 對齊）；濾掉台灣範圍外的錯誤座標
lat = pd.to_numeric(date_info['緯度'], errors='coerce').to_numpy()
lon = pd.to_numeric(date_info['經度'], errors='coerce').to_numpy()
valid = np.isfinite(lat) & np.isfinite(lon) & (lat > 20) & (lat < 26.5) & (lon > 118) & (lon < 122.5)
acc_rad = np.radians(np.c_[np.where(valid, lat, 0.0), np.where(valid, lon, 0.0)])

def _nearest(tree_pts):
    d, i = BallTree(tree_pts, metric='haversine').query(acc_rad, k=1)
    return d[:, 0] * R_KM, i[:, 0]

d_h, _   = _nearest(hosp_rad)     # 最近醫院距離(km) = 送醫可及性
d_c, _   = _nearest(city_rad)     # 最近城鎮距離(km) = 城鄉
d_p, i_p = _nearest(pop_rad)      # 最近有人口地點 → 取其人口

spatial_raw = pd.DataFrame({
    'dist_hospital_band': np.where(valid, d_h, np.nan),
    'dist_city_band':     np.where(valid, d_c, np.nan),
    'nearest_pop_band':   np.where(valid, pop_val[i_p], np.nan),
}, index=rbind_data.index)
print(spatial_raw.describe().T[['mean', '50%', 'max']])

def jenks_bin_train(series, train_positions, n_classes=5, sample=20000, random_state=42):
    """只用 train 補值並估 Jenks 內部分界，再套用到全體；輸出皆為類別字串。"""
    v = pd.to_numeric(series, errors='coerce')
    tr = v.iloc[train_positions].dropna()
    if tr.empty:
        raise ValueError(f'{series.name} 的 train 資料全為缺值')
    v = v.fillna(tr.median())
    tr = v.iloc[train_positions]
    k = min(n_classes, tr.nunique())
    if k < 2:
        return pd.Series('L1', index=v.index, dtype=object)
    s = tr.sample(min(len(tr), sample), random_state=random_state) if len(tr) > sample else tr
    try:
        breaks = jenkspy.jenks_breaks(s.values, n_classes=k)
    except TypeError:
        breaks = jenkspy.jenks_breaks(s.values, nb_class=k)
    breaks = sorted(set(float(b) for b in breaks))
    if len(breaks) < 3:
        return pd.Series('L1', index=v.index, dtype=object)
    breaks[0], breaks[-1] = -np.inf, np.inf
    labels = [f"L{i+1}" for i in range(len(breaks) - 1)]
    return pd.cut(v, bins=breaks, labels=labels, include_lowest=True).astype(str)

GEO_MAPPER_COLS = list(spatial_raw.columns)
spatial_df = pd.DataFrame(index=rbind_data.index)
for _col in GEO_MAPPER_COLS:
    spatial_df[_col] = jenks_bin_train(spatial_raw[_col], train_pos, n_classes=5)

print('train-only spatial Jenks 完成:', GEO_MAPPER_COLS)
display(spatial_df.apply(lambda s: s.value_counts().sort_index()))


In [ ]:
# ==== 地理特徵加入點雲/MCA；所有表示只 fit train ====
from utils.preprocess import PARTY_TIME_COLS
from sklearn.neighbors import NearestNeighbors

BASE_MAPPER_COLS = [c for c in rbind_data.columns if c not in PARTY_TIME_COLS]
mapper_select_lst = BASE_MAPPER_COLS + GEO_MAPPER_COLS
mapper_feature_df = pd.concat(
    [rbind_data[BASE_MAPPER_COLS], spatial_df[GEO_MAPPER_COLS]], axis=1
).astype(str)
_FORBIDDEN_LABEL_COLS = {'死亡', '受傷', '死亡受傷人數', 'color_for_plot', 'y'}
assert not (_FORBIDDEN_LABEL_COLS & set(mapper_feature_df.columns)), 'Mapper 輸入含標籤欄位'
mapper_train_source = mapper_feature_df.iloc[train_pos]
mapper_test_source = mapper_feature_df.iloc[test_pos]

# one-hot vocabulary 與 MCA 都只由 train 決定
mapper_train_df = pd.get_dummies(mapper_train_source, dtype=np.float32)
mapper_test_df = pd.get_dummies(mapper_test_source, dtype=np.float32).reindex(
    columns=mapper_train_df.columns, fill_value=0)
mapper_train = mapper_train_df.to_numpy()
mapper_test = mapper_test_df.to_numpy()

mca = prince.MCA(n_components=5, n_iter=30, copy=True, check_input=True, random_state=42)
mca.fit(mapper_train_source)
lens_train = mca.transform(mapper_train_source).to_numpy()
lens_test = mca.transform(mapper_test_source).to_numpy()

_unseen = {
    c: int((~mapper_test_source[c].isin(mapper_train_source[c].unique())).sum())
    for c in mapper_select_lst
}
_unseen = {c: n for c, n in _unseen.items() if n > 0}
print('Mapper 欄位數:', len(mapper_select_lst), '| point cloud:', mapper_train.shape, mapper_test.shape)
print('MCA lens:', lens_train.shape, lens_test.shape, '| test-only 類別:', _unseen or '無')


In [ ]:
overlap = 2
interval = 10

mapper_algo = MapperAlgorithm(
    cover=CubicalCover(n_intervals=interval, overlap_frac=overlap / 10),
    clustering=FailSafeClustering(AgglomerativeClustering(n_clusters=2, linkage='ward')),
    n_jobs=10)
_res = mapper_algo.fit_transform(mapper_train, lens_train)
graph = _res[0] if isinstance(_res, tuple) else _res
if not isinstance(graph, nx.Graph):
    graph = getattr(mapper_algo, 'graph_', graph)
    if isinstance(graph, tuple): graph = graph[0]
nodes = list(graph.nodes())
if not nodes:
    raise ValueError('Mapper 沒有產生任何 node')
print('train 建圖：nodes =', len(nodes), '| edges =', graph.number_of_edges())

# ==== train graph 上的 node-level 拓樸量；完全不使用死亡標籤 ====
node_size = {nd: len(graph.nodes[nd]['ids']) for nd in nodes}
_all_local_ids = np.concatenate([
    np.asarray(graph.nodes[nd]['ids'], dtype=int) for nd in nodes
])
assert _all_local_ids.min() >= 0 and _all_local_ids.max() < len(train_pos)

# cycle_member：屬於至少一個含 3+ nodes 的 biconnected component。
# 這比 2-core 更嚴格，不會把連接兩個 cycles 的 bridge path 誤標成 cycle。
_cycle_nodes = set()
for _block in nx.biconnected_components(graph):
    if len(_block) >= 3:
        _cycle_nodes.update(_block)
node_cycle_member = {nd: int(nd in _cycle_nodes) for nd in nodes}

node_coreness = nx.core_number(graph)
node_betweenness = nx.betweenness_centrality(graph, normalized=True, weight=None)
node_clustering = nx.clustering(graph, weight=None)
_articulation_nodes = set(nx.articulation_points(graph))
node_articulation = {nd: int(nd in _articulation_nodes) for nd in nodes}
_bridge_edges = list(nx.bridges(graph))
_bridge_nodes = {nd for edge in _bridge_edges for nd in edge}
node_bridge_endpoint = {nd: int(nd in _bridge_nodes) for nd in nodes}

def _node_distance_to_sources(sources):
    """Unweighted graph distance；無法到達來源的 component 使用 max(finite)+1。"""
    if not sources:
        return {nd: 1.0 for nd in nodes}
    finite = nx.multi_source_dijkstra_path_length(graph, sources, weight=None)
    unreachable_value = float(max(finite.values(), default=0) + 1)
    return {nd: float(finite.get(nd, unreachable_value)) for nd in nodes}

node_distance_to_cycle = _node_distance_to_sources(_cycle_nodes)

# main core：含最大 observation node 的 component 中，coreness 最高的 nodes。
_anchor = max(nodes, key=lambda nd: node_size[nd])
_main_component = set(nx.node_connected_component(graph, _anchor))
_main_core_k = max(node_coreness[nd] for nd in _main_component)
_main_core_nodes = {nd for nd in _main_component if node_coreness[nd] == _main_core_k}
node_distance_to_main_core = _node_distance_to_sources(_main_core_nodes)

print('cycle nodes:', len(_cycle_nodes),
      '| articulation nodes:', len(_articulation_nodes),
      '| bridges:', len(_bridge_edges),
      '| main core k:', _main_core_k, '| main core nodes:', len(_main_core_nodes))

# ==== node → train observation 聚合 ====
# binary=any；distance=min；coreness/betweenness=max；clustering=membership mean。
topo_cycle_member = np.zeros(_n, dtype=np.int8)
topo_distance_to_cycle = np.full(_n, np.inf)
topo_coreness = np.full(_n, -np.inf)
topo_betweenness = np.full(_n, -np.inf)
topo_clustering_sum = np.zeros(_n, dtype=np.float64)
topo_clustering_coefficient = np.full(_n, np.nan)
topo_articulation_point = np.zeros(_n, dtype=np.int8)
topo_bridge_endpoint = np.zeros(_n, dtype=np.int8)
topo_distance_to_main_core = np.full(_n, np.inf)
membership_count = np.zeros(_n, dtype=np.int16)

for nd in nodes:
    loc = np.asarray(graph.nodes[nd]['ids'], dtype=int)  # train-local positions
    orig = train_pos[loc]                                # original/global positions
    topo_cycle_member[orig] = np.maximum(topo_cycle_member[orig], node_cycle_member[nd])
    topo_distance_to_cycle[orig] = np.minimum(
        topo_distance_to_cycle[orig], node_distance_to_cycle[nd])
    topo_coreness[orig] = np.maximum(topo_coreness[orig], node_coreness[nd])
    topo_betweenness[orig] = np.maximum(topo_betweenness[orig], node_betweenness[nd])
    topo_clustering_sum[orig] += node_clustering[nd]
    topo_articulation_point[orig] = np.maximum(
        topo_articulation_point[orig], node_articulation[nd])
    topo_bridge_endpoint[orig] = np.maximum(
        topo_bridge_endpoint[orig], node_bridge_endpoint[nd])
    topo_distance_to_main_core[orig] = np.minimum(
        topo_distance_to_main_core[orig], node_distance_to_main_core[nd])
    membership_count[orig] += 1

if (membership_count[train_pos] == 0).any():
    raise ValueError('部分 train observations 沒有 Mapper node membership')
topo_clustering_coefficient[train_pos] = (
    topo_clustering_sum[train_pos] / membership_count[train_pos]
)
_raw_topology_arrays = [
    topo_distance_to_cycle, topo_coreness, topo_betweenness,
    topo_clustering_coefficient, topo_distance_to_main_core,
]
assert all(np.isfinite(arr[train_pos]).all() for arr in _raw_topology_arrays)

# AgglomerativeClustering 沒有 predict：使用可重現的兩階段 out-of-sample 延伸。
# 先在 5D MCA lens 找鄰近 train 候選，再以原始 one-hot 點雲距離重排；
# test 只繼承最近 train observation 的拓樸摘要；test 不參與 graph statistics。
CANDIDATE_K = min(20, len(train_pos))
ASSIGN_CHUNK = 5000
_lens_nn = NearestNeighbors(
    n_neighbors=CANDIDATE_K, algorithm='kd_tree', metric='euclidean', n_jobs=-1
).fit(lens_train)

_is_train = np.zeros(_n, dtype=bool)
_is_train[train_pos] = True
for start in range(0, len(test_pos), ASSIGN_CHUNK):
    stop = min(start + ASSIGN_CHUNK, len(test_pos))
    candidates = _lens_nn.kneighbors(lens_test[start:stop], return_distance=False)
    # one-hot Hamming distance；只在少量 lens 候選內計算，避免全資料暴力搜尋
    point_dist = np.count_nonzero(
        mapper_train[candidates] != mapper_test[start:stop, None, :], axis=2)
    nearest_local = candidates[np.arange(stop - start), point_dist.argmin(axis=1)]
    source_pos = train_pos[nearest_local]
    target_pos = test_pos[start:stop]
    assert _is_train[source_pos].all() and not _is_train[target_pos].any()
    topo_cycle_member[target_pos] = topo_cycle_member[source_pos]
    topo_distance_to_cycle[target_pos] = topo_distance_to_cycle[source_pos]
    topo_coreness[target_pos] = topo_coreness[source_pos]
    topo_betweenness[target_pos] = topo_betweenness[source_pos]
    topo_clustering_coefficient[target_pos] = topo_clustering_coefficient[source_pos]
    topo_articulation_point[target_pos] = topo_articulation_point[source_pos]
    topo_bridge_endpoint[target_pos] = topo_bridge_endpoint[source_pos]
    topo_distance_to_main_core[target_pos] = topo_distance_to_main_core[source_pos]

assert all(np.isfinite(arr).all() for arr in _raw_topology_arrays)


07/20/2026 03:40:11 PM core WARNING: Unable to perform clustering on local chart: Found array with 1 sample(s) (shape=(1, 163)) while a minimum of 2 is required by AgglomerativeClustering.
07/20/2026 03:40:31 PM core WARNING: Unable to perform clustering on local chart: Found array with 1 sample(s) (shape=(1, 163)) while a minimum of 2 is required by AgglomerativeClustering.
07/20/2026 03:40:34 PM core WARNING: Unable to perform clustering on local chart: Found array with 1 sample(s) (shape=(1, 163)) while a minimum of 2 is required by AgglomerativeClustering.
07/20/2026 03:40:41 PM core WARNING: Unable to perform clustering on local chart: Found array with 1 sample(s) (shape=(1, 163)) while a minimum of 2 is required by AgglomerativeClustering.
07/20/2026 03:40:41 PM core WARNING: Unable to perform clustering on local chart: Found array with 1 sample(s) (shape=(1, 163)) while a minimum of 2 is required by AgglomerativeClustering.
07/20/2026 03:40:47 PM core WARNING: Unable to perform 

train 建圖：nodes = 1165 | edges = 12190
cycle nodes: 1005 | articulation nodes: 49 | bridges: 54 | main core k: 25 | main core nodes: 29


In [37]:
# ==== 數值拓樸量只用 train 估 Jenks；binary 特徵保留 0/1 ====
topo_distance_to_cycle_band = jenks_bin_train(
    pd.Series(topo_distance_to_cycle, index=rbind_data.index,
              name='topo_distance_to_cycle'), train_pos, n_classes=5)
topo_coreness_band = jenks_bin_train(
    pd.Series(topo_coreness, index=rbind_data.index,
              name='topo_coreness'), train_pos, n_classes=5)
topo_betweenness_band = jenks_bin_train(
    pd.Series(topo_betweenness, index=rbind_data.index,
              name='topo_betweenness'), train_pos, n_classes=5)
topo_clustering_coefficient_band = jenks_bin_train(
    pd.Series(topo_clustering_coefficient, index=rbind_data.index,
              name='topo_clustering_coefficient'), train_pos, n_classes=5)
topo_distance_to_main_core_band = jenks_bin_train(
    pd.Series(topo_distance_to_main_core, index=rbind_data.index,
              name='topo_distance_to_main_core'), train_pos, n_classes=5)

TOPOLOGY_COLUMNS = [
    # 'topo_cycle_member',
    # 'topo_distance_to_cycle_band',
    'topo_coreness_band',
    'topo_betweenness_band',
    # 'topo_clustering_coefficient_band',
    # 'topo_articulation_point',
    # 'topo_bridge_endpoint',
    'topo_distance_to_main_core_band',
]
topo_df = pd.DataFrame({
    # 'topo_cycle_member': topo_cycle_member,
    # 'topo_distance_to_cycle_band': topo_distance_to_cycle_band,
    'topo_coreness_band': topo_coreness_band,
    'topo_betweenness_band': topo_betweenness_band,
    # 'topo_clustering_coefficient_band': topo_clustering_coefficient_band,
    # 'topo_articulation_point': topo_articulation_point,
    # 'topo_bridge_endpoint': topo_bridge_endpoint,
    'topo_distance_to_main_core_band': topo_distance_to_main_core_band,
}, index=rbind_data.index)

assert topo_df.columns.tolist() == TOPOLOGY_COLUMNS
assert not topo_df.isna().any().any()
assert not ({'死亡', '受傷', '死亡受傷人數'} & set(topo_df.columns))
# for _binary_col in ['topo_cycle_member', 'topo_articulation_point', 'topo_bridge_endpoint']:
#     assert set(topo_df[_binary_col].unique()).issubset({0, 1})
# print('topology features:', topo_df.shape)
# display(topo_df.describe(include='all').T)


In [38]:
# ==== 合併回完整檔並依時間排序、存檔 ====
base = rbind_data.copy()
base['死亡'] = death.values
base = pd.concat([base, spatial_df], axis=1)
base['發生日期'] = date_info['發生日期'].values
base['發生時間'] = date_info['發生時間'].values

order = _order_index
base = base.loc[order].reset_index(drop=True)
topo = topo_df.loc[order].reset_index(drop=True)
assert base.index.equals(topo.index)
assert topo.columns.tolist() == TOPOLOGY_COLUMNS
full_topo = pd.concat([base, topo], axis=1)
full_notopo = base.copy()
assert full_topo[base.columns].equals(full_notopo)
assert full_topo['死亡'].equals(full_notopo['死亡'])
assert not any(c.startswith('occ_') for c in full_topo.columns)

os.makedirs('./Data/FullData', exist_ok=True)
full_topo.to_csv('./Data/FullData/full_with_topo.csv', index=False)
full_notopo.to_csv('./Data/FullData/full_no_topo.csv', index=False)
print('已存檔:', full_topo.shape, full_notopo.shape)
print('topology columns:', topo.columns.tolist())


已存檔: (313933, 45) (313933, 42)
topology columns: ['topo_coreness_band', 'topo_betweenness_band', 'topo_distance_to_main_core_band']


In [48]:
# ==== 訓練：加 vs 不加拓樸特徵 ====
DROP = [c for c in ['發生日期', '發生時間'] if c in full_topo.columns]
X_topo,   y_topo   = models_new.get_train_test_data(full_topo.drop(columns=DROP).copy())
X_notopo, y_notopo = models_new.get_train_test_data(full_notopo.drop(columns=DROP).copy())
print('X_notopo:', X_notopo.shape, '| X_topo:', X_topo.shape)

SEED = 42
TRAIN_UNDER_RATIO = 1
ENCODING = 'onehot'
algos = [('xgboost', models_new.xgboost_cm_gridsearch),
         ('logistic', models_new.logistic_cm_gridsearch),
         ('svc', models_new.linear_svc_cm_gridsearch)]

# for tag, X, y in [('full_notopo', X_notopo, y_notopo)]:
for tag, X, y in [('full_topo', X_topo, y_topo)]:
    for algo_name, fn in algos:
        print(f'[{tag}] {algo_name} start')
        t = time.time()
        yv, sc, idx = fn(X, y, random_state=SEED, encoding=ENCODING, train_under_ratio=TRAIN_UNDER_RATIO)
        elapsed = time.time() - t
        save_dir = f"../Models/ModelPerformanceSeed/{tag}/{algo_name}"
        os.makedirs(save_dir, exist_ok=True)
        with open(f"{save_dir}/full.pkl", "wb") as f:
            pickle.dump({'y': yv, 'decision_scores': sc, 'indices': idx, 'elapsed_time': elapsed}, f)
        print(f'[{tag}] {algo_name} done in {elapsed:.1f}s')


X_notopo: (313933, 39) | X_topo: (313933, 42)
[full_topo] xgboost start


/Users/wangqiqian/opt/anaconda3/envs/TrafficTDApython/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/wangqiqian/opt/anaconda3/envs/TrafficTDApython/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Best parameters: {'clf__subsample': 0.8, 'clf__n_estimators': 400, 'clf__max_depth': 5, 'clf__learning_rate': 0.2, 'clf__colsample_bytree': 0.6}
[full_topo] xgboost done in 6.8s
[full_topo] logistic start


/Users/wangqiqian/opt/anaconda3/envs/TrafficTDApython/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/wangqiqian/opt/anaconda3/envs/TrafficTDApython/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


Best parameters: {'clf__penalty': 'l2', 'clf__C': 0.01}
[full_topo] logistic done in 9.3s
[full_topo] svc start


/Users/wangqiqian/opt/anaconda3/envs/TrafficTDApython/lib/python3.9/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/Users/wangqiqian/opt/anaconda3/envs/TrafficTDApython/lib/python3.9/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(
/Users/wangqiqian/opt/anaconda3/envs/TrafficTDApython/lib/python3.9/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/wangqiqian/opt/anaconda3/envs/TrafficTDApython/lib/python3.9/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/Users/wangqiqian/op

Best parameters: {'clf__loss': 'hinge', 'clf__C': 0.01}
[full_topo] svc done in 4.3s


In [49]:
# ==== 比較：加 vs 不加拓樸特徵 ====
rows = []
for algo_name, _ in algos:
    for tag in ['full_notopo', 'full_topo']:
        m = evaluate.metrics_from_pkl(f"../Models/ModelPerformanceSeed/{tag}/{algo_name}/full.pkl", threshold='youden')
        rows.append((f"{algo_name} | {tag}", m))
display(evaluate.build_table(rows))


,pr_auc,roc_auc,balanced_acc,recall,precision,f1,accuracy,threshold,n_test
method,,,,,,,,,
xgboost | full_notopo,0.0414,0.8352,0.7749,0.7676,0.0158,0.0309,0.7821,0.5798,94180
xgboost | full_topo,0.0415,0.8337,0.7688,0.7887,0.0141,0.0276,0.7491,0.4569,94180
logistic | full_notopo,0.0353,0.8267,0.7513,0.7746,0.0128,0.0251,0.7282,0.4522,94180
logistic | full_topo,0.0361,0.8288,0.7554,0.7676,0.0134,0.0263,0.7432,0.4616,94180
svc | full_notopo,0.0385,0.8306,0.7553,0.7817,0.0129,0.0254,0.7291,-0.2332,94180
svc | full_topo,0.0398,0.8327,0.7583,0.7183,0.0159,0.0312,0.7979,0.0013,94180
